In [2]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import pywt

In [22]:
IN_PATH  = "../data/NF-UNSW-NB15-v3.csv"
wavelet_name = 'cmor3.5-1.0'
max_samples = 5000
scales = np.arange(1, 128)
random_seed = 42

In [29]:
df = pd.read_csv(IN_PATH)

In [30]:
IAT_FEATURES = [
    "SRC_TO_DST_IAT_MIN",
    "SRC_TO_DST_IAT_AVG",
    "SRC_TO_DST_IAT_MAX",
    "SRC_TO_DST_IAT_STDDEV",
    "DST_TO_SRC_IAT_MIN",
    "DST_TO_SRC_IAT_AVG",
    "DST_TO_SRC_IAT_MAX",
    "DST_TO_SRC_IAT_STDDEV"
]

In [40]:
df['flow_id'] = df.index

In [41]:
df_0 = df[df["Label"] == 0].sample(n=max_samples, random_state=random_seed)
df_1 = df[df["Label"] == 1].sample(n=max_samples, random_state=random_seed)

In [42]:
df_sample = pd.concat([df_0, df_1]).sample(frac=1, random_state=42)

In [44]:
rows = []

In [45]:
for _, row in df_sample.iterrows():
    signal = row[IAT_FEATURES].values.astype(float)
    
    if np.any(np.isnan(signal)) or np.std(signal) == 0:
        continue
    
    signal = (signal - signal.mean()) / signal.std()
    
    coeffs, _ = pywt.cwt(signal, scales, wavelet_name)
    coeff_mag = np.abs(coeffs)
    features = coeff_mag.mean(axis=1)
    
    out = {"label": int(row["Label"]), "flow_id": row["flow_id"]}
    for i, val in enumerate(features):
        out[f"cwt_scale_{i}"] = val
    
    rows.append(out)

In [46]:
OUT_PATH = "cwt_binary_label.csv"

In [47]:
df_cwt_binary = pd.DataFrame(rows)
df_cwt_binary.to_csv(OUT_PATH, index=False)
df_cwt_binary.head()

,label,flow_id,cwt_scale_0,cwt_scale_1,cwt_scale_2,cwt_scale_3,cwt_scale_4,cwt_scale_5,cwt_scale_6,cwt_scale_7,...,cwt_scale_117,cwt_scale_118,cwt_scale_119,cwt_scale_120,cwt_scale_121,cwt_scale_122,cwt_scale_123,cwt_scale_124,cwt_scale_125,cwt_scale_126
0,1,366645,0.026187,0.131546,0.246531,0.531713,0.392564,0.239554,0.223322,0.261458,...,0.058610,0.058858,0.059104,0.059350,0.059595,0.059839,0.060081,0.060323,0.060564,0.060804
1,0,1170943,0.023581,0.305857,0.328660,0.580817,0.411046,0.215342,0.082396,0.011906,...,0.062034,0.062296,0.062557,0.062817,0.063076,0.063334,0.063591,0.063847,0.064102,0.064356
2,0,542038,0.023344,0.314943,0.322206,0.575210,0.407437,0.214143,0.083256,0.013400,...,0.063935,0.064205,0.064474,0.064742,0.065009,0.065275,0.065540,0.065804,0.066066,0.066328
3,0,540480,0.024536,0.268924,0.330943,0.597990,0.430702,0.230303,0.092187,0.013262,...,0.059172,0.059422,0.059671,0.059919,0.060166,0.060412,0.060657,0.060901,0.061145,0.061387
4,0,817994,0.024392,0.234902,0.318332,0.600377,0.427567,0.215873,0.076700,0.053621,...,0.061576,0.061836,0.062096,0.062354,0.062611,0.062867,0.063122,0.063376,0.063629,0.063881
